# Quantify endodyogeny

**Purpose.** Classify intracellular replication states from pathogen size and summarize their frequencies across experimental groups.

**Recommended use.** Use when parasite division states are represented by approximately log2-scaled object-size classes.

**Primary outputs.** Per-object replication-state assignments and group-level proportions.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.submodules.analyze_endodyogeny`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_endodyogeny)

```python
analyze_endodyogeny(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.submodules import analyze_endodyogeny

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.submodules.analyze_endodyogeny`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_endodyogeny)


#### Paths

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.

#### Measurements

- **`tables`** *(optional)* — (list) - Measurement tables read from each plate's database and merged into one analysis frame. Only 'cell', 'nucleus', 'pathogen', 'cytoplasm', and 'png_list' are merged. Any other table, including 'organelle', is loaded but omitted from the merged result without a warning. Default ['cell', 'nucleus', 'pathogen', 'cytoplasm'].
- **`compartment`** *(optional)* — (str) - Prefix used by per-object measurement columns, so 'pathogen' selects pathogen_area and pathogen_channel_1_percentile_95. It must match the object type contained in the table; otherwise the run stops and reports the unresolved area and intensity columns. Default 'pathogen'.
- **`nuclei_limit`** *(optional)* — (int, bool, or None) - Cap on nuclei per cell, applied when the per-object tables are merged. None disables the filter, True keeps only single-nucleus cells, and an integer N keeps cells with N or fewer. Cells over the cap are dropped from the merged table entirely. Do not pass False: it is interpreted as 0 and removes every cell, leaving an empty analysis rather than raising an error. Default None.
- **`pathogen_limit`** *(optional)* — (int, bool, or None) - Maximum pathogens per cell. True or 1 = single pathogen only; None or False = no limit; int = custom limit. Default varies by module (1, 3, 10 or 1000 depending on the factory that fills it), so check the module's own settings rather than assuming one value.

#### Plate Layout & Controls

- **`cell_types`** *(optional)* — (list) - Names of the host cell lines in the experiment, e.g. ['HeLa']. Each name is written into the host_cells column and folded into the combined condition label used for grouping and plotting; the list is positionally paired with cell_plate_metadata, which says which wells hold each one. Default ['HeLa'].
- **`cell_plate_metadata`** *(optional)* — (list of lists) - Wells occupied by each entry of cell_types, with one inner list per cell type in the same order, for example [['c2','c3'],['c4']]. Each identifier must start with 'c' (column) or 'r' (row); invalid identifiers are skipped without an exception and those wells receive no host_cells label. Because 'condition' combines the labels that are present, a typographical error changes the comparison without raising an error. Default None.
- **`pathogen_types`** *(optional)* — (list) - Names given to each pathogen condition on the plate, e.g. ['wt','ku80']. Element i is written into the pathogen column for every well listed in pathogen_plate_metadata[i] and folded into the combined condition label used for grouping and plotting. Must match pathogen_plate_metadata in length and order; None skips pathogen annotation. Default ['pathogen_1', 'pathogen_2'] for the dataset builders, ['pc'] for the control-based paths, None where types are not used.
- **`pathogen_plate_metadata`** *(optional)* — (list of lists) - Well locations of each pathogen condition, one inner list per entry in pathogen_types. Every item must be a row or column ID string such as 'c1' or 'r3'; anything else is silently ignored and those wells stay unannotated. Ranges like 'c2-c11' are not expanded - list each row/column. Do not leave it None while pathogen_types is set: annotation is not skipped, every row is labelled with the first pathogen_types entry. Defaults: None in the plot-from-db settings, [['c1','c2','c3'],['c4','c5','c6']] for recruitment analysis.
- **`treatments`** *(optional)* — (list) - Names of the drug or treatment conditions in the experiment, e.g. ['dmso','lovastatin']. Each name is written into the treatment column and folded into the combined condition label used for grouping and plotting; positionally paired with treatment_plate_metadata (or treatment_loc), which lists the wells for each. Default ['cm','lovastatin'].
- **`treatment_plate_metadata`** *(optional)* — (list of lists) - Wells that received each treatment, with one inner list per treatment in the same order, for example [['r1','r2','r3'],['r4','r5','r6']]. Entries must start with 'r' (row) or 'c' (column); other entries are ignored and receive no treatment label. Unlisted wells remain in the output, and their condition values contain only the available cell, pathogen, or treatment labels. Default None.
- **`group_column`** *(optional)* — (str) - Column whose values become the experimental conditions compared against each other; 'condition' is the combined host-cell / pathogen / treatment label built from the plate-metadata maps. Point it at 'pathogen' or 'treatment' to compare on one factor alone. Rows with no value here are dropped before anything is counted. Default 'condition'.
- **`level`** *(optional)* — (str) - Which level the run reports, on BOTH the fitted and the permutation side. 'both' answers the gRNA and the gene question separately, writes results_grna.csv and results_gene.csv, puts every row in results.csv marked by a 'level' column, and corrects each family independently with multiple_testing_method -- a gene fraction is the sum of its guides' fractions, so one combined design would be collinear and one shared correction would count the same wells twice. 'grna' or 'gene' reports one of them. Under inference='nonparametric' the guide pass runs whatever you pick, because a gene's regressor IS the sum of its guides', so 'gene' means the primary table reports genes rather than that guides were skipped. Disabled only for fitted mixed models, which nest guides within genes and answer both at once. Proportion plots use the same key for a different question: 'object' pools objects, 'well' averages by well, 'plate' averages by plate. Default 'both' for regression and 'object' for proportions.
- **`change_plate`** *(optional)* — (bool) - Relabel each source directory as plate1, plate2, ... instead of trusting the plate ID stored in its database. Use it when several plates were written under the same name, which would otherwise let two plates' fields pool into one threshold and one well. Default False.

#### Plot

- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') preserve the relative visibility of intensity differences; 'gray' resembles the raw single-channel microscope image. Any registered matplotlib name is accepted, with an '_r' suffix to reverse it. Default 'inferno' for image plots and 'viridis' for plate heatmaps.

#### Endodyogeny Size Proxy (Legacy)

- **`class_column`** *(optional)* — (str) - Column containing the per-object class label used by class-proportion analysis. Missing values are filled with 0 rather than dropping the corresponding rows; selecting an incorrect column can therefore assign class zero to every object without raising an error. The value is also appended to the condition when group_by_class is enabled. Default 'test'.
- **`group_by_class`** *(optional)* — (bool) - Whether the endodyogeny condition labels are split by class before proportions are computed: on, the condition string has the class_column value appended, so each condition-class combination becomes its own group; off, classes are pooled within a condition. It changes what the bars count, not how the statistics are weighted. Default False.
- **`um_per_px`** *(required)* — (float or None) - Physical size of one pixel, used to convert the endodyogeny area column into square microns before binning. Set it and max_area, min_area_bin and every reported area are in microns; leave it None and they stay in pixels, which makes numbers from objectives of different magnification incomparable. Default 0.1.
- **`min_area_bin`** *(optional)* — (int) - Width of the smallest area bin in the endodyogeny histogram, and therefore the resolution at which small parasites are distinguished from one another. Expressed in the same unit as max_area, so it follows um_per_px when a scale is set. Too small a value produces sparse noisy bins; too large merges real division states. Default 500.
- **`max_area`** *(optional)* — (int) - Upper area cutoff applied before the endodyogeny bins are built; larger objects are excluded. The cutoff is applied after um_per_px scaling and therefore uses square micrometres when a scale is set or square pixels when it is None. The default is effectively unbounded. Default 1000000000.
- **`max_bins`** *(optional)* — (int or None) - Maximum number of area bins in the endodyogeny histogram. None derives the bin count from the data range and min_area_bin; an integer truncates the range so that the largest objects are pooled into the final bin. Set an integer when plates with different size ranges must share one axis. Default None.

#### Advanced

- **`verbose`** *(optional)* — (bool) - Print the resolved settings table, channel and model choices per object type, row counts per table, and object counts after each filter. It only adds console output; enable it to identify which stage produced an unexpected object count. The default is True for mask, UMAP, screen analysis, barcode mapping and Cellpose training, and False for measure, plotting helpers and regression.
- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Paths
    # Required settings
    'src': 'path',

    # Measurements
    # Optional settings
    'tables': ['cell', 'nucleus', 'pathogen', 'cytoplasm'],
    'compartment': 'pathogen',
    'nuclei_limit': 10,
    'pathogen_limit': 1,

    # Plate Layout & Controls
    # Optional settings
    'cell_types': ['Hela'],
    'cell_plate_metadata': None,
    'pathogen_types': ['pc'],
    'pathogen_plate_metadata': [['c1'], ['c2']],
    'treatments': None,
    'treatment_plate_metadata': None,
    'group_column': 'condition',
    'level': 'object',
    'change_plate': False,

    # Plot
    # Optional settings
    'cmap': 'viridis',

    # Endodyogeny Size Proxy (Legacy)
    # Required settings
    'um_per_px': 0.1,
    # Optional settings
    'class_column': 'predictions',
    'group_by_class': False,
    'min_area_bin': 500,
    'max_area': 1000000000,
    'max_bins': None,

    # Advanced
    # Optional settings
    'verbose': False,
    'save': False,
}

In [ ]:
analyze_endodyogeny(settings)

## Outputs and next steps

Per-object replication-state assignments and group-level proportions.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)